# PALLAS surrogates on the feature store

Three programs, one store between them. The feature pipeline writes simulation runs as rows, the training pipeline reads them through feature views and registers models, the inference pipeline deploys a model and searches it. Each step runs on its own; they share nothing but the feature store and the model registry.

Data: the organisers' public 10 % sample of the PALLAS pack (Smilei PIC simulations), read over HTTPS from a pinned commit. See `README.md` for the source and citation.

## 1. Feature pipeline

Read the pack, apply the data card's rules, write four feature groups. `--out` writes the exact frames to parquet without a cluster; `--insert` writes them to the feature store.

In [ ]:
%run feature_pipeline.py --out pallas_fg

In [ ]:
import pandas as pd

b = pd.read_parquet("pallas_fg/pallas_campaign_b.parquet")
b.loc[b.injected, ["config", "p_max", "cn2_max", "l_inj", "dip_frac", "x_of", "e_med_mev", "de_mad", "q_end"]].head()

On a cluster, the same frames go into the feature store. Run it from the project terminal or as a job; laptop writes on a lossy line are slow.

In [ ]:
# !hops job deploy pallas_features feature_pipeline.py --args "--insert" --run --wait
# %run feature_pipeline.py --insert

## 2. Training pipeline

Two feature views, two models:

- `pallas_direct_b`: profile knobs to energy, spread, charge. Injected campaign B rows inside the organisers' reference train split. A Gaussian process per target.
- `pallas_inverse_a`: beam plus focal position back to `p_1`, `a_0`, `c_N2`. Injected rows of campaign A's random scan. Gradient boosting per target.

R² is on a 20 % holdout, on the scorer's scales, with a linear baseline on the same split.

In [ ]:
%run training_pipeline.py --local pallas_fg --models models

With a cluster: create the feature views, train from them, register both models with their views.

In [ ]:
# %run training_pipeline.py

## 3. Inference pipeline

Deploy `pallas_direct_b` with `predictor.py`, then let scikit-optimize find profile settings for a 200 MeV bunch with at least 20 pC. Locally, the same loop calls the pickled model instead of the deployment.

In [ ]:
%run inference_pipeline.py --local models/pallas_direct_b/model.pkl --fg-dir pallas_fg --target-mev 200 --min-charge-pc 20

In [ ]:
# %run inference_pipeline.py --target-mev 200 --min-charge-pc 20

The nearest real simulation in the sample is a sanity check, not a validation: the answer is one Smilei run at the proposed settings, written back to `pallas_campaign_b`. The next model trains on it.

In [ ]:
b[b.injected & b.e_med_mev.between(190, 210)].sort_values("de_mad")[["config", "e_med_mev", "de_mad", "q_pc"]].head(3)